Celda 1 — Verificar que todo está preinstalado


In [ ]:
import sys
print(sys.version)
import geopandas, rasterio, fiona, pyproj, shapely, huggingface_hub
print("geopandas", geopandas.__version__)
print("rasterio", rasterio.__version__)
print("TODO OK")

Celda 2 — Descargar el GeoPackage desde Hugging Face


In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

RAW = Path('/content/data/neon')
RAW.mkdir(parents=True, exist_ok=True)

path = hf_hub_download(
    repo_id="weecology/neon-tree-crowns-dta",
    filename="neon_crowns_dta.gpkg",
    repo_type="dataset",
    local_dir=str(RAW),
)
print("Descargado:", path)
print("Tamaño:", Path(path).stat().st_size / 1e6, "MB")

Celda 3 — Explorar: ¿qué sitios tienen más hand_bbox?


In [ ]:
import geopandas as gpd
from pathlib import Path

gdf = gpd.read_file('/content/data/neon/neon_crowns_dta.gpkg')
print("Total coronas:", len(gdf))
print("Columnas:", gdf.columns.tolist())
print("\nPor crown_source:")
print(gdf["crown_source"].value_counts())
print("\nPor sitio (top 15):")
print(gdf["siteID"].value_counts().head(15))
print("\nhand_bbox por sitio (top 10):")
hb = gdf[gdf["crown_source"] == "hand_bbox"]
print(hb["siteID"].value_counts().head(10))
print("\nalgorithmic por sitio (top 10):")
alg = gdf[gdf["crown_source"] == "algorithmic"]
print(alg["siteID"].value_counts().head(10))

Celda 4 — Guardar el .gpkg en Drive (para no re-descargar)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import shutil
dst = '/content/drive/MyDrive/neon_crowns_dta.gpkg'
shutil.copy('/content/data/neon/neon_crowns_dta.gpkg', dst)
print("Guardado en Drive:", dst)

Celda 5 — Resumen de sitios candidatos (para decidir)


In [ ]:
# Tabla cruzada: por sitio, cuántas algorithmic vs hand_bbox
pivot = gdf.pivot_table(index='siteID', columns='crown_source',
                        values='crown_id', aggfunc='count', fill_value=0)
pivot['total'] = pivot.sum(axis=1)
pivot = pivot.sort_values('total', ascending=False)
print(pivot.head(20))